# Recast

This notebook recasts the EmoBank likert scores into pairwise comparisons. 

In [11]:
import pandas as pd
import numpy as np
from itertools import combinations
import itertools
import choix

In [3]:
df = pd.read_csv("Data/individual_reader_ratings.csv")
df

,id,V,A,D
0,Acephalous-Cant-believe_4_47,3,3,3
1,Acephalous-Cant-believe_4_47,3,3,3
2,Acephalous-Cant-believe_4_47,3,5,3
3,Acephalous-Cant-believe_4_47,3,3,4
4,Acephalous-Cant-believe_4_47,3,3,3
...,...,...,...,...
53050,SemEval_1499,3,3,3
53051,SemEval_1499,3,3,3
53052,SemEval_1499,3,3,3
53053,SemEval_1499,2,4,2


In [4]:
items = pd.unique(df['id']).tolist()

# Generate combinations of length 2
# distinct_pairs is an iterator; wrap in list() to see all at once
pairs = list(combinations(items, 2))

print(f"Total pairs: {len(pairs)}")

Total pairs: 55624878


In [5]:
def get_scores(df, item1_id, item2_id, dimension):
    """
    Returns the scores for two items on a given dimension in df.
    """
    item1_scores = df[df['id'] == item1_id][dimension].tolist()
    item2_scores = df[df['id'] == item2_id][dimension].tolist()
    return item1_scores, item2_scores

# Example
get_scores(df, pairs[0][0], pairs[0][1], 'V')

([3, 3, 3, 3, 3], [2, 3, 3, 3, 3])

In [6]:
def get_pairwise_probability(scores_a, scores_b):
    """
    Calculates P(A > B) using an all-pairs comparison.
    Handles ties by splitting the point (0.5).
    Args:
        scores_a (list or array-like): Scores for item A.
        scores_b (list or array-like): Scores for item B.
    Returns:
        float: Probability that item A is rated higher than item B.
    """
    # Convert to numpy arrays for speed
    a = np.array(scores_a)
    b = np.array(scores_b)
    
    # Create a grid of comparisons using broadcasting
    # This creates a matrix where matrix[i, j] is a[i] - b[j]
    diff_matrix = a[:, np.newaxis] - b
    
    # Count wins (A > B), losses (A < B), and ties (A == B)
    wins = np.sum(diff_matrix > 0)
    ties = np.sum(diff_matrix == 0)
    total_comparisons = a.size * b.size
    
    # Calculate probability: Wins + half of ties
    prob_a_better = (wins + (0.5 * ties)) / total_comparisons
    
    return prob_a_better

# Example Usage
item_a_scores = [4, 5, 4, 5, 3] # Generally high
item_b_scores = [2, 3, 2, 4, 1] # Generally low

prob = get_pairwise_probability(item_a_scores, item_b_scores)

print(f"Probability that Item A is rated higher than Item B: {prob:.2f}")
# Result > 0.5 implies A > B

Probability that Item A is rated higher than Item B: 0.90


In [8]:
def estimate_batch_size(example_row, sample_n=1000, target_fraction=0.25):
    """
    Estimate a safe batch_size given an example row dict.
    target_fraction: fraction of free RAM to use (0.2-0.5 typical).
    Args:
        example_row (dict): Example row to estimate memory usage.
        sample_n (int): Number of rows to sample for estimation.
        target_fraction (float): Fraction of available memory to use.
    Returns:
        int: Estimated batch size.
    """
    try:
        import psutil
    except Exception:
        raise RuntimeError("psutil required: pip3 install psutil")
    import pandas as pd

    sample = pd.DataFrame([example_row] * sample_n)
    bytes_per_row = sample.memory_usage(deep=True).sum() / sample_n
    avail = psutil.virtual_memory().available
    batch_size = max(1, int((avail * target_fraction) // bytes_per_row))
    return batch_size

# Example usage
example_row = {
    'Dimension': 'V',
    'Item_A': items[0],
    'Item_B': items[1],
    'Probability_A_greater_B': 0.5
}

estimated_batch_size = estimate_batch_size(example_row, sample_n=1000, target_fraction=0.25) 

print("Estimated batch_size:", estimated_batch_size)

Estimated batch_size: 247860603


In [9]:
# Streamed pairwise computation: use combinations() iterator and write batches to disk
from itertools import combinations
import os

dims = ['V', 'A', 'D']
id_scores = {item: grp[dims].to_dict('list') for item, grp in df.groupby('id')}

outfile = "Data/pairwise_probs.csv"
if os.path.exists(outfile):
    os.remove(outfile)

batch = []
batch_size = estimated_batch_size 
write_header = True

for a, b in combinations(items, 2):
    for dim in dims:
        prob = get_pairwise_probability(id_scores[a][dim], id_scores[b][dim])
        batch.append({
            'Dimension': dim,
            'Item_A': a,
            'Item_B': b,
            'Probability_A_greater_B': prob
        })
    if len(batch) >= batch_size:
        pd.DataFrame(batch).to_csv(outfile, mode='a', header=write_header, index=False)
        write_header = False
        batch.clear()

# write any remaining rows
if batch:
    pd.DataFrame(batch).to_csv(outfile, mode='a', header=write_header, index=False)

# Process Data and Split

In [3]:
df_probs = pd.read_csv("Data/pairwise_probs.csv")
df_probs

,Dimension,Item_A,Item_B,Probability_A_greater_B
0,V,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_83_354,0.60
1,A,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_83_354,0.52
2,D,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_83_354,0.50
3,V,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_355_499,0.30
4,A,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_355_499,0.60
...,...,...,...,...
166874629,A,SemEval_1497,SemEval_1499,0.50
166874630,D,SemEval_1497,SemEval_1499,0.60
166874631,V,SemEval_1498,SemEval_1499,0.60
166874632,A,SemEval_1498,SemEval_1499,0.32


In [4]:
# Make a new column with a 1/0 indicator if Probability_A_greater_B > 0.5
df_probs['A_better_indicator'] = (df_probs['Probability_A_greater_B'] > 0.5).astype(int)
df_probs

,Dimension,Item_A,Item_B,Probability_A_greater_B,A_better_indicator
0,V,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_83_354,0.60,1
1,A,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_83_354,0.52,1
2,D,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_83_354,0.50,0
3,V,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_355_499,0.30,0
4,A,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_355_499,0.60,1
...,...,...,...,...,...
166874629,A,SemEval_1497,SemEval_1499,0.50,0
166874630,D,SemEval_1497,SemEval_1499,0.60,1
166874631,V,SemEval_1498,SemEval_1499,0.60,1
166874632,A,SemEval_1498,SemEval_1499,0.32,0


In [6]:
# Generate 3 separate files for V, A, D
dims = ['V', 'A', 'D']
for dim in dims:
    dim_df = df_probs[df_probs["Dimension"] == dim].copy()
    dim_df.to_csv(f"Data/pairwise_probs_{dim}.csv")

# Generate BT-Scores

In [7]:
pairwise_probs_V = pd.read_csv("Data/pairwise_probs_V.csv")
pairwise_probs_V

,Unnamed: 0,Dimension,Item_A,Item_B,Probability_A_greater_B,A_better_indicator
0,0,V,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_83_354,0.60,1
1,3,V,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_355_499,0.30,0
2,6,V,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_500_515,0.50,0
3,9,V,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_517_626,0.50,0
4,12,V,Acephalous-Cant-believe_4_47,Acephalous-Cant-believe_627_817,0.50,0
...,...,...,...,...,...,...
55624873,166874619,V,SemEval_1496,SemEval_1498,0.20,0
55624874,166874622,V,SemEval_1496,SemEval_1499,0.28,0
55624875,166874625,V,SemEval_1497,SemEval_1498,0.40,0
55624876,166874628,V,SemEval_1497,SemEval_1499,0.50,0


In [ ]:
# Bradley-Terry via MM algorithm using pairwise_probs_V['A_better_indicator']
# Assumes pairwise_probs_V exists in the notebook.

# Aggregate pairwise counts (n_ab) and wins for Item_A (wins_A)
pair_counts = (
    pairwise_probs_V[['Item_A', 'Item_B', 'A_better_indicator']]
    .groupby(['Item_A', 'Item_B'], sort=False)
    .agg(n_ab=('A_better_indicator', 'size'), wins_A=('A_better_indicator', 'sum'))
    .reset_index()
)

# Items and index mapping
items = pd.Index(pd.unique(pd.concat([pair_counts['Item_A'], pair_counts['Item_B']])))
idx = {item: i for i, item in enumerate(items)}
m = len(items)

# Wins per item (s_i)
wins_as_A = pair_counts.groupby('Item_A')['wins_A'].sum().reindex(items, fill_value=0)
counts_as_B = pair_counts.groupby('Item_B')['n_ab'].sum().reindex(items, fill_value=0)
wins_A_when_B = pair_counts.groupby('Item_B')['wins_A'].sum().reindex(items, fill_value=0)
wins_as_B = counts_as_B - wins_A_when_B
s = (wins_as_A + wins_as_B).to_numpy(dtype=float)

# Prepare index arrays for vectorized denominator accumulation
ia = pair_counts['Item_A'].map(idx).to_numpy(dtype=np.int64)
ib = pair_counts['Item_B'].map(idx).to_numpy(dtype=np.int64)
n_ab = pair_counts['n_ab'].to_numpy(dtype=float)

# MM iterations
pi = np.ones(m, dtype=float)  # positive strengths
tol = 1e-6
max_iter = 1000

for it in range(max_iter):
    # denom contributions per pair: n_ab / (pi_i + pi_j)
    contrib = n_ab / (pi[ia] + pi[ib])
    # accumulate to den[i] and den[j] using bincount
    den = np.bincount(np.concatenate([ia, ib]),
                      weights=np.concatenate([contrib, contrib]),
                      minlength=m).astype(float)
    # update
    pi_new = np.where(den > 0, s / den, 0.0)
    # avoid zeros
    pi_new = np.maximum(pi_new, 1e-12)
    # normalize to mean 1 for stability
    pi_new /= pi_new.mean()
    if np.max(np.abs(pi_new - pi) / pi) < tol:
        pi = pi_new
        break
    pi = pi_new

# Bradley-Terry scores: log-strengths centered to mean zero
bt_scores = np.log(pi)
bt_scores -= bt_scores.mean()

bt_df = pd.DataFrame({'Item': items, 'BT_score': bt_scores})
bt_df = bt_df.sort_values('BT_score', ascending=False).reset_index(drop=True)

bt_df.head(50)  # show top 50 items

In [12]:
# Aggregate pairwise counts (n_ab) and wins for Item_A (wins_A)
pair_counts = (
    pairwise_probs_V[['Item_A', 'Item_B', 'A_better_indicator']]
    .groupby(['Item_A', 'Item_B'], sort=False)
    .agg(n_ab=('A_better_indicator', 'size'), wins_A=('A_better_indicator', 'sum'))
    .reset_index()
)

# Items and index mapping
items = pd.Index(pd.unique(pd.concat([pair_counts['Item_A'], pair_counts['Item_B']])))
idx = {item: i for i, item in enumerate(items)}
m = len(items)

# Prepare data in choix format: list of (winner, loser) tuples
comparisons = []
for _, row in pair_counts.iterrows():
    i_idx = idx[row['Item_A']]
    j_idx = idx[row['Item_B']]
    wins_a = int(row['wins_A'])
    wins_b = int(row['n_ab'] - wins_a)
    
    # Add wins for A
    comparisons.extend([(i_idx, j_idx)] * wins_a)
    # Add wins for B
    comparisons.extend([(j_idx, i_idx)] * wins_b)

# Fit Bradley-Terry model using choix
bt_params = choix.ilsr_pairwise(m, comparisons, alpha=0.0)

# Bradley-Terry scores: already log-scale, center to mean zero
bt_scores = bt_params - bt_params.mean()

bt_df = pd.DataFrame({'Item': items, 'BT_score': bt_scores})
bt_df = bt_df.sort_values('BT_score', ascending=False).reset_index(drop=True)

bt_df.head(50)  # show top 50 items

ValueError: stationary distribution could not be computed. Perhaps the Markov chain has more than one absorbing class?